# 因子获取+数据处理

In [1]:
from atrader import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels import  regression
import statsmodels.api as sm
import scipy.stats as st
import math

## 因子获取

In [ ]:
# 市值因子
mkv = get_factor_by_factor(factor='market_cap', target_list=list(get_code_list('hs300',date='2020-01-01').code), 
                           begin_date='2020-01-01', end_date='2024-12-31')

# 技术指标因子
factor_macd = get_factor_by_factor(factor='MACD_DIFF', target_list=list(get_code_list('hs300',date='2020-01-01').code), 
                              begin_date='2020-01-01', end_date='2024-12-31')
factor_rsi10 = get_factor_by_factor(factor='RSI10', target_list=list(get_code_list('hs300',date='2020-01-01').code), 
                              begin_date='2020-01-01', end_date='2024-12-31')
factor_mtm = get_factor_by_factor(factor='MTM', target_list=list(get_code_list('hs300',date='2020-01-01').code), 
                              begin_date='2020-01-01', end_date='2024-12-31')

#反转/波动率因子（超跌 & 风险溢价）
factor_bias5 = get_factor_by_factor(factor='BIAS5', target_list=list(get_code_list('hs300',date='2020-01-01').code), 
                              begin_date='2020-01-01', end_date='2024-12-31')
factor_volt20 = get_factor_by_factor(factor='VOLT20', target_list=list(get_code_list('hs300',date='2020-01-01').code), 
                              begin_date='2020-01-01', end_date='2024-12-31')
factor_mdd20 = get_factor_by_factor(factor='MDD20', target_list=list(get_code_list('hs300',date='2020-01-01').code), 
                              begin_date='2020-01-01', end_date='2024-12-31')

#估值因子
factor_pe = get_factor_by_factor(factor='pe_ratio_ttm', target_list=list(get_code_list('hs300',date='2020-01-01').code), 
                              begin_date='2020-01-01', end_date='2024-12-31')
factor_pb = get_factor_by_factor(factor='pb_ratio_ttm', target_list=list(get_code_list('hs300',date='2020-01-01').code), 
                              begin_date='2020-01-01', end_date='2024-12-31')
factor_ev = get_factor_by_factor(factor='ev_to_ebitda_ttm', target_list=list(get_code_list('hs300',date='2020-01-01').code), 
                              begin_date='2020-01-01', end_date='2024-12-31')

# 成长类因子
factor_inc = get_factor_by_factor(factor='inc_revenue_ttm', target_list=list(get_code_list('hs300',date='2020-01-01').code), 
                              begin_date='2020-01-01', end_date='2024-12-31')
factor_return = get_factor_by_factor(factor='return_on_equity_ttm', target_list=list(get_code_list('hs300',date='2020-01-01').code), 
                              begin_date='2020-01-01', end_date='2024-12-31')
factor_du = get_factor_by_factor(factor='du_profit_margin_ttm', target_list=list(get_code_list('hs300',date='2020-01-01').code), 
                              begin_date='2020-01-01', end_date='2024-12-31')

# 行为因子
factor_vol20 = get_factor_by_factor(factor='VOL20', target_list=list(get_code_list('hs300',date='2020-01-01').code), 
                              begin_date='2020-01-01', end_date='2024-12-31')
factor_qtyr = get_factor_by_factor(factor='QTYR_5_20', target_list=list(get_code_list('hs300',date='2020-01-01').code), 
                              begin_date='2020-01-01', end_date='2024-12-31')


In [3]:
mkv.head(5)

,date,SZSE.000001,SZSE.000002,SZSE.000063,SZSE.000069,SZSE.000100,SZSE.000157,SZSE.000166,SZSE.000333,SZSE.000338,SZSE.000413,SZSE.000415,SZSE.000423,SZSE.000425,SZSE.000538,SZSE.000568,SZSE.000596,...,SSE.601988,SSE.601989,SSE.601992,SSE.601997,SSE.601998,SSE.603019,SSE.603156,SSE.603160,SSE.603259,SSE.603260,SSE.603288,SSE.603501,SSE.603799,SSE.603833,SSE.603899,SSE.603986,SSE.603993
0,2020-01-02,3.27378e+11,3.67998e+11,1.49862e+11,6.38155e+10,6.18250e+10,5.40774e+10,1.28956e+11,4.16243e+11,1.28211e+11,1.94255e+10,2.36249e+10,2.32439e+10,4.54353e+10,1.14085e+11,1.25163e+11,6.90939e+10,...,1.09512e+12,1.21079e+11,4.04688e+10,3.08931e+10,3.03885e+11,3.19970e+10,3.09624e+10,9.86660e+10,1.50090e+11,2.84026e+10,2.91640e+11,1.28781e+11,4.32871e+10,4.90255e+10,4.45372e+10,7.04858e+10,9.43887e+10
1,2020-01-03,3.33394e+11,3.62234e+11,1.54513e+11,6.35694e+10,6.29072e+10,5.38413e+10,1.29206e+11,4.05863e+11,1.30274e+11,1.98267e+10,2.35012e+10,2.37933e+10,4.52786e+10,1.13817e+11,1.25661e+11,6.70141e+10,...,1.09218e+12,1.22447e+11,4.02552e+10,3.07644e+10,3.03396e+11,3.21590e+10,3.08359e+10,9.83561e+10,1.46224e+11,2.84026e+10,2.84646e+11,1.25231e+11,4.76018e+10,4.89666e+10,4.37276e+10,7.07651e+10,9.67646e+10
2,2020-01-06,3.31259e+11,3.56131e+11,1.55147e+11,6.37335e+10,6.23661e+10,5.23457e+10,1.28205e+11,3.98479e+11,1.20516e+11,1.98267e+10,2.35012e+10,2.39633e+10,4.36335e+10,1.12488e+11,1.25412e+11,6.80464e+10,...,1.08629e+12,1.21763e+11,4.01484e+10,3.04104e+10,2.99970e+11,3.23391e+10,3.05195e+10,9.68887e+10,1.41867e+11,2.88716e+10,2.79812e+11,1.22208e+11,5.04495e+10,4.78994e+10,4.39944e+10,7.25631e+10,9.69806e+10
3,2020-01-07,3.32811e+11,3.58956e+11,1.54724e+11,6.43077e+10,6.18250e+10,5.25818e+10,1.29206e+11,4.04400e+11,1.21309e+11,2.06289e+10,2.35012e+10,2.38587e+10,4.41036e+10,1.13127e+11,1.25969e+11,6.81975e+10,...,1.09512e+12,1.21763e+11,4.02552e+10,3.06356e+10,3.02417e+11,3.26362e+10,3.08042e+10,9.80189e+10,1.44013e+11,2.89279e+10,2.86239e+11,1.19099e+11,4.95326e+10,4.89918e+10,4.52364e+10,6.83185e+10,9.54686e+10
4,2020-01-08,3.23303e+11,3.58052e+11,1.50961e+11,6.25851e+10,6.38542e+10,5.15679e+10,1.26201e+11,4.06003e+11,1.17183e+11,2.01705e+10,2.27590e+10,2.33224e+10,4.34769e+10,1.11811e+11,1.24284e+11,6.95774e+10,...,1.08629e+12,1.26779e+11,3.89739e+10,3.00564e+10,2.98502e+11,3.27442e+10,3.04035e+10,9.53346e+10,1.43112e+11,2.84589e+10,2.86698e+11,1.19781e+11,4.99533e+10,4.89078e+10,4.42152e+10,6.85497e+10,9.63326e+10


In [6]:
# 检查市值数据中是否有市值为0的记录
# 将数据转换为长格式：每行是一只股票在某一天的市值
mkv_long = mkv.melt(id_vars='date', var_name='code', value_name='mkt_value')
# 过滤出市值为0的记录
zero_mkv = mkv_long[mkv_long['mkt_value'] == 0]
# 查看结果
print(zero_mkv)

Empty DataFrame
Columns: [date, code, mkt_value]
Index: []


## 3sigma去极值+标准化

In [4]:
def winsorize_and_standardize(df: pd.DataFrame) -> pd.DataFrame:
    """
    对宽表因子数据，按每列（股票）做3σ去极值和标准化。
    """
    # 分离date列
    df_copy = df.copy()
    date_col = df_copy.iloc[:, 0]
    
    # 其余列数据
    data = df_copy.iloc[:, 1:]
    
    # 逐列做3sigma去极值和标准化
    for col in data.columns:
        series = data[col]
        mean = series.mean()
        std = series.std()
        upper = mean + 3 * std
        lower = mean - 3 * std
        
        clipped = series.clip(lower=lower, upper=upper)
        
        mean_c = clipped.mean()
        std_c = clipped.std()
        if std_c == 0:
            data[col] = 0
        else:
            data[col] = (clipped - mean_c) / std_c
    
    # 合并date列和处理后的数据列
    df_processed = pd.concat([date_col, data], axis=1)
    return df_processed


In [5]:
mkv_processed = winsorize_and_standardize(mkv)

In [6]:
mkv_processed = winsorize_and_standardize(mkv)
factor_macd_processed = winsorize_and_standardize(factor_macd)
factor_rsi10_processed = winsorize_and_standardize(factor_rsi10)
factor_mtm_processed = winsorize_and_standardize(factor_mtm)
factor_bias5_processed = winsorize_and_standardize(factor_bias5)
factor_volt20_processed = winsorize_and_standardize(factor_volt20)
factor_mdd20_processed = winsorize_and_standardize(factor_mdd20)
factor_pe_processed = winsorize_and_standardize(factor_pe)
factor_pb_processed = winsorize_and_standardize(factor_pb)
factor_ev_processed = winsorize_and_standardize(factor_ev)
factor_inc_processed = winsorize_and_standardize(factor_inc)
factor_return_processed = winsorize_and_standardize(factor_return)
factor_du_processed = winsorize_and_standardize(factor_du)
factor_vol20_processed = winsorize_and_standardize(factor_vol20)
factor_qtyr_processed = winsorize_and_standardize(factor_qtyr)

In [ ]:
# #保存处理后的数据到CSV文件
# pd.DataFrame(mkv_processed).to_csv('mkv_processed.csv', index=False)
# pd.DataFrame(factor_macd_processed).to_csv('factor_macd_processed.csv', index=False)
# pd.DataFrame(factor_rsi10_processed).to_csv('factor_rsi10_processed.csv', index=False)
# pd.DataFrame(factor_mtm_processed).to_csv('factor_mtm_processed.csv', index=False)
# pd.DataFrame(factor_bias5_processed).to_csv('factor_bias5_processed.csv', index=False)
# pd.DataFrame(factor_volt20_processed).to_csv('factor_volt20_processed.csv', index=False)
# pd.DataFrame(factor_mdd20_processed).to_csv('factor_mdd20_processed.csv', index=False)
# pd.DataFrame(factor_pe_processed).to_csv('factor_pe_processed.csv', index=False)
# pd.DataFrame(factor_pb_processed).to_csv('factor_pb_processed.csv', index=False)
# pd.DataFrame(factor_ev_processed).to_csv('factor_ev_processed.csv', index=False)
# pd.DataFrame(factor_inc_processed).to_csv('factor_inc_processed.csv', index=False)
# pd.DataFrame(factor_return_processed).to_csv('factor_return_processed.csv', index=False)
# pd.DataFrame(factor_du_processed).to_csv('factor_du_processed.csv', index=False)
# pd.DataFrame(factor_vol20_processed).to_csv('factor_vol20_processed.csv', index=False)
# pd.DataFrame(factor_qtyr_processed).to_csv('factor_qtyr_processed.csv', index=False)


In [8]:
mkv_processed.head(5)

,date,SZSE.000001,SZSE.000002,SZSE.000063,SZSE.000069,SZSE.000100,SZSE.000157,SZSE.000166,SZSE.000333,SZSE.000338,SZSE.000413,SZSE.000415,SZSE.000423,SZSE.000425,SZSE.000538,SZSE.000568,SZSE.000596,...,SSE.601988,SSE.601989,SSE.601992,SSE.601997,SSE.601998,SSE.603019,SSE.603156,SSE.603160,SSE.603259,SSE.603260,SSE.603288,SSE.603501,SSE.603799,SSE.603833,SSE.603899,SSE.603986,SSE.603993
0,2020-01-02,0.61924,1.75276,0.08388,1.17103,-1.12520,-0.63391,0.94836,-0.41190,0.20236,2.11393,2.83719,-0.75949,-0.88546,0.12417,-1.93936,-2.04292,...,0.17322,2.06557,2.26720,2.78517,0.75203,-1.33090,0.12084,1.68248,-1.12349,-1.14585,-0.56213,-0.52929,-1.06174,-0.75503,-0.27352,-0.51402,-1.01588
1,2020-01-03,0.69869,1.68547,0.25599,1.15571,-1.06622,-0.65037,0.96843,-0.52901,0.29517,2.22654,2.79234,-0.67400,-0.89500,0.11139,-1.93292,-2.13251,...,0.15682,2.18719,2.23059,2.74164,0.74117,-1.31998,0.09037,1.67213,-1.16560,-1.14585,-0.61672,-0.59574,-0.94694,-0.75809,-0.32293,-0.50054,-0.93254
2,2020-01-06,0.67050,1.61423,0.27946,1.16593,-1.09571,-0.75461,0.88812,-0.61233,-0.14388,2.22654,2.79234,-0.64754,-0.99514,0.04809,-1.93614,-2.08804,...,0.12403,2.12638,2.21229,2.62193,0.66514,-1.30785,0.01420,1.62309,-1.21306,-1.13517,-0.65445,-0.65233,-0.87118,-0.81356,-0.30665,-0.41379,-0.92496
3,2020-01-07,0.69100,1.64721,0.26382,1.20168,-1.12520,-0.73815,0.96843,-0.54552,-0.10819,2.45175,2.79234,-0.66382,-0.96653,0.07852,-1.92895,-2.08153,...,0.17322,2.12638,2.23059,2.69811,0.71945,-1.28783,0.08275,1.66086,-1.18969,-1.13388,-0.60428,-0.71053,-0.89557,-0.75678,-0.23084,-0.61858,-0.97799
4,2020-01-08,0.56541,1.63666,0.12456,1.09443,-1.01461,-0.80882,0.72750,-0.52744,-0.29380,2.32306,2.52321,-0.74728,-1.00468,0.01583,-1.95073,-2.02209,...,0.12403,2.57233,2.01095,2.50222,0.63256,-1.28055,-0.01373,1.57116,-1.19950,-1.14457,-0.60070,-0.69776,-0.88438,-0.76115,-0.29317,-0.60742,-0.94769


## 中性化

In [9]:
import atrader as at
import statsmodels.api as sm

In [10]:
# 申万一级行业
shenwan_industry = {
'申万农林牧渔':'SWNLMY',
'申万采掘':'SWCJ',
'申万化工新材料2':'SWHGXLC2',
'申万钢铁':'SWGT',
'申万有色金属':'SWYSJS',
'申万电子':'SWDZ',
'申万家用电器':'SWJYDQ',
'申万食品饮料':'SWSPYL',
'申万纺织服饰':'SWFZFS',
'申万轻工制造':'SWQGZZ',
'申万医药生物':'SWYYSW',
'申万公用事业':'SWGYSY',
'申万交通运输':'SWJTYS',
'申万房地产':'SWFDC',
'申万石油贸易':'SWSYMY',
'申万信息服务1':'SWXXFW1',
'申万综合':'SWZH1',
'申万建筑材料':'SWJZCL',
'申万建筑装饰':'SWJZZS',
'申万电气设备2':'SWDQSB2',
'申万国防军工':'SWGFJG',
'申万计算机':'SWJSJ',
'申万传媒':'SWCM',
'申万通信':'SWTX',
'申万银行':'SWYH',
'申万非银金融':'SWFYJR',
'申万汽车':'SWQC',
'申万机械设备':'SWJXSB'
}

In [11]:
at.get_code_list('SWCJ')

,code,name,block_name,weight,date
0,szse.000552,甘肃能化,gsnh,0.0,2024-05-13
1,szse.000571,新大洲A,xdza,0.0,2024-05-13
2,szse.000629,钒钛股份,ftgf,0.0,2024-05-13
3,szse.000655,金岭矿业,jlky,0.0,2024-05-13
4,szse.000723,美锦能源,mjny,0.0,2024-05-13
5,szse.000762,西藏矿业,xzky,0.0,2024-05-13
6,szse.000780,ST平能,stpn,0.0,2024-05-13
7,szse.000833,粤桂股份,yggf,0.0,2024-05-13
8,szse.000923,河钢资源,hgzy,0.0,2024-05-13
9,szse.000937,冀中能源,jzny,0.0,2024-05-13


In [ ]:
def build_industry_dummy_matrix(df_factors, shenwan_industry_dict):
    """
    构造行业哑变量矩阵，适用于因子数据的DataFrame，df_factors包含date列（非索引），
    其余列为股票代码，股票代码转换为小写处理。

    参数：
        df_factors: DataFrame，含 'date' 列，其他列为股票代码（如 SZSE.000001）
        shenwan_industry_dict: dict，{行业中文名: 行业代码}

    返回：
        df_ind_dummy: DataFrame，index为股票代码（小写），columns为行业名，值为0或1
    """
    # 将因子数据列名中除date外全部转小写股票代码
    stock_codes = [col.lower() for col in df_factors.columns if col != 'date']

    # 初始化哑变量矩阵
    industry_names = list(shenwan_industry_dict.keys())
    df_ind_dummy = pd.DataFrame(0, index=stock_codes, columns=industry_names)

    # 遍历每个行业，调用at.get_code_list获取行业股票代码列表
    for industry_name, industry_code in shenwan_industry_dict.items():
        try:
            # 获取行业中所有股票代码
            df_ind = at.get_code_list(industry_code)
            # 小写转换
            codes_in_industry = df_ind['code'].str.lower()
            # 找出存在于因子列中的股票代码
            valid_codes = set(codes_in_industry) & set(stock_codes)
            # 标记哑变量矩阵
            df_ind_dummy.loc[list(valid_codes), industry_name] = 1
            print(f"行业 '{industry_name}': 包含股票数 {len(valid_codes)}")
        except Exception as e:
            print(f"处理行业 {industry_name} 出错：{e}")
            df_ind_dummy[industry_name] = 0

    return df_ind_dummy

In [13]:
dummy_matrix=build_industry_dummy_matrix(mkv, shenwan_industry)

行业 '申万农林牧渔': 包含股票数 4
行业 '申万采掘': 包含股票数 10
行业 '申万化工新材料2': 包含股票数 0
行业 '申万钢铁': 包含股票数 6
行业 '申万有色金属': 包含股票数 13
行业 '申万电子': 包含股票数 17
行业 '申万家用电器': 包含股票数 6
行业 '申万食品饮料': 包含股票数 10
行业 '申万纺织服饰': 包含股票数 2
行业 '申万轻工制造': 包含股票数 2
行业 '申万医药生物': 包含股票数 26
行业 '申万公用事业': 包含股票数 9
行业 '申万交通运输': 包含股票数 18
行业 '申万房地产': 包含股票数 13
行业 '申万石油贸易': 包含股票数 0
行业 '申万信息服务1': 包含股票数 10
行业 '申万综合': 包含股票数 0
行业 '申万建筑材料': 包含股票数 5
行业 '申万建筑装饰': 包含股票数 9
行业 '申万电气设备2': 包含股票数 7
行业 '申万国防军工': 包含股票数 9
行业 '申万计算机': 包含股票数 14
行业 '申万传媒': 包含股票数 9
行业 '申万通信': 包含股票数 5
行业 '申万银行': 包含股票数 24
行业 '申万非银金融': 包含股票数 37
行业 '申万汽车': 包含股票数 10
行业 '申万机械设备': 包含股票数 7


In [14]:
dummy_matrix

,申万农林牧渔,申万采掘,申万化工新材料2,申万钢铁,申万有色金属,申万电子,申万家用电器,申万食品饮料,申万纺织服饰,申万轻工制造,申万医药生物,申万公用事业,申万交通运输,申万房地产,申万石油贸易,申万信息服务1,申万综合,申万建筑材料,申万建筑装饰,申万电气设备2,申万国防军工,申万计算机,申万传媒,申万通信,申万银行,申万非银金融,申万汽车,申万机械设备
szse.000001,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
szse.000002,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
szse.000063,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
szse.000069,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
szse.000100,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
szse.000157,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
szse.000166,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
szse.000333,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
szse.000338,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
szse.000413,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [41]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

def neutralize(factor_df, dummy_matrix, mkv_df):
    """
    行业和对数市值中性化处理函数，带详细调试信息
    
    参数：
        factor_df: DataFrame，行索引为日期，列为股票代码，因子值
        dummy_matrix: DataFrame，股票代码为索引，列为行业名称，值为0/1哑变量
        mkv_df: DataFrame，行索引为日期，列为股票代码，原始市值
    
    返回：
        factor_neutralized: DataFrame，中性化后的因子，结构同factor_df
    """
    # 统一股票代码小写，确保一致
    factor_df.columns = factor_df.columns.str.lower()
    mkv_df.columns = mkv_df.columns.str.lower()
    dummy_matrix.index = dummy_matrix.index.str.lower()

    # 设置日期索引（如果有date列）
    if 'date' in factor_df.columns:
        factor_df = factor_df.set_index('date')
    if 'date' in mkv_df.columns:
        mkv_df = mkv_df.set_index('date')

    factor_df.index = pd.to_datetime(factor_df.index)
    mkv_df.index = pd.to_datetime(mkv_df.index)

    # 初始化结果DataFrame
    factor_neutralized = pd.DataFrame(index=factor_df.index, columns=factor_df.columns)

    # 对市值做对数变换，避免0或负值
    mkv_log = np.log(mkv_df.replace(0, np.nan))

    for date in factor_df.index:
        # 取当日数据
        if date not in mkv_log.index:
            print(f"日期 {date} 不在市值数据中，跳过")
            continue

        y = factor_df.loc[date].dropna()
        x_mkv = mkv_log.loc[date].dropna()

        # 找交集股票代码
        valid_codes = y.index.intersection(x_mkv.index).intersection(dummy_matrix.index)       
        y_valid = y.loc[valid_codes]
        x_mkv_valid = x_mkv.loc[valid_codes]
        X_industry = dummy_matrix.loc[valid_codes]
        X = pd.concat([X_industry, x_mkv_valid.rename('log_mkv')], axis=1)

        # 删除含NaN的行
        valid_rows = X.dropna().index.intersection(y_valid.dropna().index)
        X = X.loc[valid_rows]
        y_reg = y_valid.loc[valid_rows]

        # 线性回归
        model = LinearRegression()
        model.fit(X, y_reg)
        y_pred = model.predict(X)
        
        residual = y_reg - y_pred
        factor_neutralized.loc[date, residual.index] = residual

    print("中性化处理完成")
    return factor_neutralized


In [ ]:
neutralize(factor_pe_processed, dummy_matrix, mkv).head(5)

中性化处理完成


,szse.000001,szse.000002,szse.000063,szse.000069,szse.000100,szse.000157,szse.000166,szse.000333,szse.000338,szse.000413,szse.000415,szse.000423,szse.000425,szse.000538,szse.000568,szse.000596,szse.000625,...,sse.601988,sse.601989,sse.601992,sse.601997,sse.601998,sse.603019,sse.603156,sse.603160,sse.603259,sse.603260,sse.603288,sse.603501,sse.603799,sse.603833,sse.603899,sse.603986,sse.603993
date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2020-01-02,-0.60928,0.12799,0.45774,-0.63927,-0.73266,-0.65675,-1.17586,-0.14292,0.12098,2.5691,0.3062,-0.40749,-0.54606,0.17936,-0.01609,-0.25581,0.36083,...,-0.58532,0.66062,0.02047,1.55977,0.56644,0.56005,-1.27881,-0.80231,-0.11469,-0.83446,0.28472,3.3088,-3.72474,-0.18494,0.18494,-0.17931,0.65235
2020-01-03,-0.54663,0.09826,0.54005,-0.63228,-0.73654,-0.65551,-1.1618,-0.20379,0.19087,2.60133,0.30769,-0.39735,-0.54492,0.17411,0.02716,-0.28513,0.37141,...,-0.60045,0.61712,0.02888,1.51932,0.55708,0.57238,-1.26037,-0.8081,-0.14739,-0.83923,0.25616,3.30891,-3.9106,-0.1609,0.1609,-0.17519,0.73945
2020-01-06,-0.51012,0.08815,0.5624,-0.6175,-0.74275,-0.68752,-1.15786,-0.22057,-0.07468,2.59001,0.31217,-0.38435,-0.61852,0.14249,0.04973,-0.23344,0.4034,...,-0.57629,0.59518,0.03015,1.43623,0.51065,0.5688,-1.28607,-0.82005,-0.16246,-0.82034,0.24552,3.31278,-3.95455,-0.19789,0.19789,-0.12645,0.71721
2020-01-07,-0.54131,0.09045,0.52485,-0.6324,-0.7365,-0.71316,-1.17332,-0.17979,-0.06013,2.65298,0.27956,-0.42021,-0.62454,0.13509,0.01037,-0.27615,0.38799,...,-0.55542,0.59552,0.0345,1.46515,0.5378,0.58366,-1.29927,-0.80553,-0.17622,-0.83029,0.25767,3.32155,-3.9403,-0.20748,0.20748,-0.24587,0.67144
2020-01-08,-0.5299,0.1143,0.49717,-0.60815,-0.70101,-0.71259,-1.11612,-0.18209,-0.19266,2.64075,0.35921,-0.38565,-0.62042,0.11934,-0.0154,-0.23427,0.37812,...,-0.51957,0.59491,0.04763,1.37644,0.52297,0.62156,-1.34831,-0.79045,-0.14256,-0.82522,0.26283,3.35594,-3.94827,-0.17734,0.17734,-0.20469,0.69804
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-25,-1.57686,-3.0003,0.36212,-0.41355,0.04497,-0.53249,0.83127,0.61015,-1.62983,NaN,0.98918,0.01955,-0.437,-0.42637,-0.63421,-0.94838,-0.5435,...,1.99868,0.80305,-1.58744,-0.03957,0.4976,1.26233,2.33884,0.13012,-0.77731,0.53744,0.06072,-0.50986,0.15372,0.02723,-0.02723,0.17551,-0.90208
2024-12-26,-1.56055,-2.99456,0.4583,-0.41876,-0.02226,-0.51272,0.79779,0.60862,-1.64968,NaN,0.98877,0.03936,-0.42798,-0.46394,-0.61816,-0.95801,-0.53968,...,2.01651,0.75562,-1.53266,-0.05795,0.44186,1.2386,2.24261,0.06768,-0.74323,0.51482,0.06009,-0.57681,0.15211,0.03629,-0.03629,0.19697,-0.89193
2024-12-27,-1.56706,-3.00832,0.45927,-0.42949,0.01535,-0.57942,0.84933,0.59957,-1.65621,NaN,0.97536,0.03834,-0.44387,-0.45258,-0.62406,-0.96564,-0.53913,...,1.97933,0.7478,-1.57257,-0.03803,0.43941,1.10448,2.25085,0.10671,-0.7546,0.46584,0.06435,-0.53829,0.15518,0.03362,-0.03362,0.18039,-0.90204


In [ ]:
factor_macd_processed = winsorize_and_standardize(factor_macd)
factor_rsi10_processed = winsorize_and_standardize(factor_rsi10)
factor_mtm_processed = winsorize_and_standardize(factor_mtm)
factor_bias5_processed = winsorize_and_standardize(factor_bias5)
factor_volt20_processed = winsorize_and_standardize(factor_volt20)
factor_mdd20_processed = winsorize_and_standardize(factor_mdd20)
factor_pe_processed = winsorize_and_standardize(factor_pe)
factor_pb_processed = winsorize_and_standardize(factor_pb)
factor_ev_processed = winsorize_and_standardize(factor_ev)
factor_inc_processed = winsorize_and_standardize(factor_inc)
factor_return_processed = winsorize_and_standardize(factor_return)
factor_du_processed = winsorize_and_standardize(factor_du)
factor_vol20_processed = winsorize_and_standardize(factor_vol20)
factor_qtyr_processed = winsorize_and_standardize(factor_qtyr)

In [ ]:
macd = neutralize(factor_macd_processed, dummy_matrix, mkv)
rsi10 = neutralize(factor_rsi10_processed, dummy_matrix, mkv)
mtm = neutralize(factor_mtm_processed, dummy_matrix, mkv)
bias5 = neutralize(factor_bias5_processed, dummy_matrix, mkv)
volt20 = neutralize(factor_volt20_processed, dummy_matrix, mkv)
mdd20 = neutralize(factor_mdd20_processed, dummy_matrix, mkv)
pe = neutralize(factor_pe_processed, dummy_matrix, mkv)
pb = neutralize(factor_pb_processed, dummy_matrix, mkv)
ev = neutralize(factor_ev_processed, dummy_matrix, mkv)
inc = neutralize(factor_inc_processed, dummy_matrix, mkv)
return_ = neutralize(factor_return_processed, dummy_matrix, mkv)
du = neutralize(factor_du_processed, dummy_matrix, mkv)
vol20 = neutralize(factor_vol20_processed, dummy_matrix, mkv)
qtyr = neutralize(factor_qtyr_processed, dummy_matrix, mkv)

中性化处理完成
中性化处理完成
中性化处理完成
中性化处理完成
中性化处理完成
中性化处理完成
中性化处理完成
中性化处理完成
中性化处理完成
中性化处理完成
中性化处理完成
中性化处理完成
中性化处理完成
中性化处理完成


In [ ]:
# # 保存中性化后的因子数据到CSV文件
# macd.to_csv('macd.csv')
# rsi10.to_csv('rsi10.csv')
# mtm.to_csv('mtm.csv')
# bias5.to_csv('bias5.csv')
# volt20.to_csv('volt20.csv')
# mdd20.to_csv('mdd20.csv')
# pe.to_csv('pe.csv')
# pb.to_csv('pb.csv')
# ev.to_csv('ev.csv')
# inc.to_csv('inc.csv')
# return_.to_csv('return_.csv')
# du.to_csv('du.csv')
# vol20.to_csv('vol20.csv')
# qtyr.to_csv('qtyr.csv')

# 行情获取+数据处理

In [44]:
data = get_kdata(target_list=list(get_code_list('hs300',date='2020-01-01').code), frequency='month', fre_num=1,
                 begin_date='2020-01-01', end_date='2024-12-31',fq=1,fill_up=True,df=True,sort_by_date=False)
close = data.pivot_table(values='close',index='code',columns='time')           # 数据透视，形成收盘价dataframe
close.head(5)

time,2020-01-23 15:00:00,2020-02-28 15:00:00,2020-03-31 15:00:00,2020-04-30 15:00:00,2020-05-29 15:00:00,2020-06-30 15:00:00,2020-07-31 15:00:00,2020-08-31 15:00:00,2020-09-30 15:00:00,2020-10-30 15:00:00,2020-11-30 15:00:00,2020-12-31 15:00:00,2021-01-29 15:00:00,2021-02-26 15:00:00,2021-03-31 15:00:00,2021-04-30 15:00:00,2021-05-31 15:00:00,...,2023-08-31 15:00:00,2023-09-28 15:00:00,2023-10-31 15:00:00,2023-11-30 15:00:00,2023-12-29 15:00:00,2024-01-31 15:00:00,2024-02-29 15:00:00,2024-03-29 15:00:00,2024-04-30 15:00:00,2024-05-31 15:00:00,2024-06-28 15:00:00,2024-07-31 15:00:00,2024-08-30 15:00:00,2024-09-30 15:00:00,2024-10-31 15:00:00,2024-11-29 15:00:00,2024-12-31 15:00:00
code,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
sse.600000,8.95893,8.56426,8.01173,8.39061,8.34325,8.35114,8.62272,8.62272,7.81538,7.70718,8.37302,8.05675,8.28979,8.77253,9.14707,8.36470,8.54781,...,6.73212,6.84786,6.57781,6.60674,6.38491,6.58745,6.89609,6.87680,7.42655,8.04383,7.93773,8.40000,8.43,10.13,9.84,9.46,10.29
sse.600004,14.99183,15.12865,12.26515,15.44139,15.97890,14.89410,13.87770,15.02745,13.43782,12.39123,14.83985,13.95124,13.61554,13.01326,13.17123,12.00616,11.44337,...,11.69745,11.11208,10.50687,10.94342,9.70323,9.33613,10.05048,10.00088,10.26876,10.06040,9.40558,9.55441,9.13,10.46,9.68,9.72,9.60
sse.600009,68.05217,64.82615,59.99206,69.74903,71.41630,71.10060,67.01629,76.13844,68.54453,65.87370,78.25118,75.40098,78.72954,61.66815,57.70178,48.97177,49.32057,...,39.09569,37.76028,37.08261,35.75716,32.66778,32.98668,35.05956,36.13586,37.51114,34.63103,32.13959,34.58000,33.13,38.42,34.96,34.91,34.15
sse.600010,1.24058,1.16118,1.14133,1.07186,1.09171,1.07186,1.16118,1.15125,1.14133,1.13140,1.20088,1.16118,1.14133,1.52839,1.54824,1.45892,1.62763,...,1.79000,1.72000,1.62000,1.53000,1.46000,1.42000,1.54000,1.60000,1.60000,1.54000,1.40000,1.46000,1.44,1.73,1.71,1.92,1.86
sse.600011,4.91641,4.27118,4.26210,3.83498,3.93494,3.83498,4.39483,5.16322,5.06951,4.57287,4.81650,4.19804,3.90755,3.77636,4.13245,3.95441,3.94504,...,7.97566,7.70165,7.44722,7.54508,7.53529,8.57262,8.63133,9.17935,9.15978,8.76834,9.41422,7.95000,6.94,7.71,7.26,6.97,6.77


In [46]:
# 数据透视形成的行标签排序与前面不一致，将原始数据的排序去重复项后排序不变
code = sorted(set(list(data['code'])),key =list(data['code']).index)     
stock_close = close.loc[code]               # 形成行标签与前面一致的dataframe
stock_close.head(5)

time,2020-01-23 15:00:00,2020-02-28 15:00:00,2020-03-31 15:00:00,2020-04-30 15:00:00,2020-05-29 15:00:00,2020-06-30 15:00:00,2020-07-31 15:00:00,2020-08-31 15:00:00,2020-09-30 15:00:00,2020-10-30 15:00:00,2020-11-30 15:00:00,2020-12-31 15:00:00,2021-01-29 15:00:00,2021-02-26 15:00:00,2021-03-31 15:00:00,2021-04-30 15:00:00,2021-05-31 15:00:00,...,2023-08-31 15:00:00,2023-09-28 15:00:00,2023-10-31 15:00:00,2023-11-30 15:00:00,2023-12-29 15:00:00,2024-01-31 15:00:00,2024-02-29 15:00:00,2024-03-29 15:00:00,2024-04-30 15:00:00,2024-05-31 15:00:00,2024-06-28 15:00:00,2024-07-31 15:00:00,2024-08-30 15:00:00,2024-09-30 15:00:00,2024-10-31 15:00:00,2024-11-29 15:00:00,2024-12-31 15:00:00
code,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
szse.000001,13.56623,12.65833,11.17425,12.16072,11.54245,11.36487,11.84433,13.38924,13.46915,15.75989,17.52677,17.17162,20.50117,18.98289,19.54226,20.67875,21.65566,...,10.38900,10.45434,9.76361,9.03554,8.76485,8.83019,9.88496,9.81962,10.07164,10.37034,10.15000,10.27,10.16,12.21,11.38,11.38,11.70
szse.000002,23.66605,24.16420,20.94666,21.88579,20.98749,21.34681,21.91845,23.09037,23.72542,23.32745,25.99466,24.30119,23.53067,28.02681,25.40194,23.85243,22.60773,...,13.64000,13.08000,11.33000,11.44000,10.46000,9.60000,10.04000,9.00000,7.41000,8.25000,6.93000,7.09,6.76,9.72,9.28,8.60,7.26
szse.000063,36.85834,47.04919,40.27411,38.66503,33.92247,37.76168,36.85834,36.89234,31.31119,30.52605,32.85311,31.83147,30.45037,30.04361,27.73548,27.38547,29.74091,...,34.60393,31.90004,25.52589,25.01830,25.84802,21.43589,29.14735,27.32198,27.98575,26.31656,27.30246,27.42,24.83,31.15,30.28,31.16,40.40
szse.000069,6.33095,5.83914,5.71395,5.83020,5.22214,5.68981,6.73200,6.71322,6.36583,6.15927,6.83528,6.65689,6.17804,7.58641,9.56752,8.74128,7.83053,...,4.35000,4.23000,3.62000,3.45000,3.11000,2.95000,3.07000,2.69000,2.64000,2.59000,2.04000,2.06,1.92,2.92,2.96,3.05,2.67
szse.000100,4.21373,4.89659,3.44760,3.95188,4.44586,5.28053,5.33163,6.03002,5.23794,5.13574,5.97040,6.03002,7.31609,7.00947,7.95486,7.66528,6.97407,...,4.02424,4.00461,3.85738,4.10277,4.22055,3.97517,4.44630,4.58371,4.69168,4.20092,4.32000,3.92,3.91,4.58,5.30,4.68,5.03


In [ ]:
# #保存收盘价数据
# pd.DataFrame(stock_close).to_csv('stock_close.csv')  

In [48]:
# 收盘价转换成长格式（long format）
df = stock_close.T
df.columns.name = 'code'
df.index.name = 'date'
df = df.reset_index().melt(id_vars='date', var_name='code', value_name='close')

# 按股票计算未来1期收益率（下月收益率）
df['close_next'] = df.groupby('code')['close'].shift(-1)
df['fwd_return'] = (df['close_next'] - df['close']) / df['close']

# 设定二分类标签
df['label'] = (df['fwd_return'] > 0).astype(int)
stock_long = df

In [49]:
df.head(5)  # 查看数据

,date,code,close,close_next,fwd_return,label
0,2020-01-23 15:00:00,szse.000001,13.56623,12.65833,-0.06692,0
1,2020-02-28 15:00:00,szse.000001,12.65833,11.17425,-0.11724,0
2,2020-03-31 15:00:00,szse.000001,11.17425,12.16072,0.08828,1
3,2020-04-30 15:00:00,szse.000001,12.16072,11.54245,-0.05084,0
4,2020-05-29 15:00:00,szse.000001,11.54245,11.36487,-0.01538,0


In [ ]:
# # 读取数据
# macd = pd.read_csv('macd.csv', index_col=0, parse_dates=True)
# rsi10 = pd.read_csv('rsi10.csv', index_col=0, parse_dates=True)
# mtm = pd.read_csv('mtm.csv', index_col=0, parse_dates=True)
# bias5 = pd.read_csv('bias5.csv', index_col=0, parse_dates=True)
# volt20 = pd.read_csv('volt20.csv', index_col=0, parse_dates=True)
# mdd20 = pd.read_csv('mdd20.csv', index_col=0, parse_dates=True)
# pe = pd.read_csv('pe.csv', index_col=0, parse_dates=True)
# pb = pd.read_csv('pb.csv', index_col=0, parse_dates=True)
# ev = pd.read_csv('ev.csv', index_col=0, parse_dates=True)
# inc = pd.read_csv('inc.csv', index_col=0, parse_dates=True)
# return_ = pd.read_csv('return_.csv', index_col=0, parse_dates=True)
# du = pd.read_csv('du.csv', index_col=0, parse_dates=True)
# vol20 = pd.read_csv('vol20.csv', index_col=0, parse_dates=True)
# qtyr = pd.read_csv('qtyr.csv', index_col=0, parse_dates=True)

In [75]:
macd.head(5)

,szse.000001,szse.000002,szse.000063,szse.000069,szse.000100,szse.000157,szse.000166,szse.000333,szse.000338,szse.000413,szse.000415,szse.000423,szse.000425,szse.000538,szse.000568,szse.000596,szse.000625,...,sse.601988,sse.601989,sse.601992,sse.601997,sse.601998,sse.603019,sse.603156,sse.603160,sse.603259,sse.603260,sse.603288,sse.603501,sse.603799,sse.603833,sse.603899,sse.603986,sse.603993
date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2020-01-02,0.06927,1.82558,0.26369,-0.02552,0.77161,-0.03279,-0.49990,-0.49370,1.55212,-3.69678,-0.37712,-0.09679,0.81977,0.22077,-0.14193,0.50861,0.24365,...,-0.45372,-0.21331,1.10715,0.45842,-0.35059,-0.95453,-0.13121,-0.17112,-0.00628,-0.42615,-0.32253,0.27771,0.29572,0.52156,-0.52156,0.96519,0.48282
2020-01-03,0.16268,1.76671,0.33765,-0.05921,0.75003,-0.05166,-0.48184,-0.53612,1.60888,-3.77538,-0.40825,-0.06897,0.88820,0.18347,-0.12849,0.53172,0.22241,...,-0.47739,-0.15150,1.28686,0.37804,-0.37615,-0.98781,-0.05381,-0.04872,-0.15237,-0.49642,-0.37170,0.12501,0.36892,0.55421,-0.55421,1.03690,0.45689
2020-01-06,0.26201,1.66737,0.41124,-0.01689,0.69547,-0.11408,-0.47284,-0.56302,1.40270,-3.86256,-0.40797,-0.00867,0.83082,0.13632,-0.09100,0.59608,0.23152,...,-0.48066,-0.14913,1.26247,0.23799,-0.40822,-1.03567,-0.02464,-0.01307,-0.29951,-0.56396,-0.42256,-0.04418,0.39929,0.52937,-0.52937,1.15572,0.34387
2020-01-07,0.31517,1.57427,0.44534,-0.00449,0.63933,-0.18596,-0.45631,-0.55211,1.22980,-3.94427,-0.45119,-0.02519,0.77405,0.08186,-0.10431,0.58977,0.24334,...,-0.44322,-0.13678,1.25539,0.14277,-0.42297,-1.06792,-0.01323,0.08483,-0.41431,-0.63190,-0.43167,-0.20414,0.43063,0.49929,-0.49929,1.06406,0.26174
2020-01-08,0.34061,1.55791,0.44477,0.01562,0.71044,-0.23970,-0.43402,-0.57619,0.94712,-3.92773,-0.47837,-0.03054,0.72620,0.03610,-0.12462,0.62972,0.28040,...,-0.39858,0.05700,1.27881,0.03268,-0.39962,-1.01634,-0.03924,0.08293,-0.47419,-0.62042,-0.43923,-0.28651,0.43115,0.49982,-0.49982,1.03134,0.16978


In [ ]:
# 因子名称列表
factorlist = ['macd', 'rsi10', 'mtm', 'bias5', 'volt20', 'mdd20', 'pe',
              'pb', 'ev', 'inc', 'return_', 'du', 'vol20', 'qtyr']

# 把所有因子变量整理成字典
factor_data_dict = {
    'macd': macd,
    'rsi10': rsi10,
    'mtm': mtm,
    'bias5': bias5,
    'volt20': volt20,
    'mdd20': mdd20,
    'pe': pe,
    'pb': pb,
    'ev': ev,
    'inc': inc,
    'return_': return_,
    'du': du,
    'vol20': vol20,
    'qtyr': qtyr
}

# 创建长格式
for factor in factorlist:
    df = factor_data_dict[factor].T.copy()  # 行转列
    df.index.name = 'code'                 # 行是股票代码
    df.columns.name = 'date'              # 列是日期
    df_long = df.reset_index().melt(id_vars='code', var_name='date', value_name=factor)
  
    # 赋值给命名变量
    globals()[f"{factor}_long"] = df_long


In [79]:
macd_long.head(5)

,code,date,macd
0,szse.000001,2020-01-02 00:00:00,0.06927
1,szse.000002,2020-01-02 00:00:00,1.82558
2,szse.000063,2020-01-02 00:00:00,0.26369
3,szse.000069,2020-01-02 00:00:00,-0.02552
4,szse.000100,2020-01-02 00:00:00,0.77161


In [80]:
# 初始化合并表为第一个因子的长格式
merged_factors = macd_long.copy()

# 依次将其余因子合并进来
for factor in factorlist[1:]:  # 从第二个因子开始
    df_long = globals()[f"{factor}_long"]
    merged_factors = pd.merge(
        merged_factors,
        df_long,
        on=['date','code'],
        how='outer'  # 如果有缺失也保留
    )

# 查看结果
print(merged_factors.head())
print(merged_factors.columns)

          code                 date     macd    rsi10      mtm    bias5   volt20    mdd20       pe       pb       ev      inc  return_       du    vol20     qtyr
0  szse.000001  2020-01-02 00:00:00  0.06927  0.21470  0.12755  0.09612  0.10389 -0.34733 -0.60928 -0.46948  0.97365 -0.34224 -0.56676 -1.58745 -0.58688  0.31619
1  szse.000002  2020-01-02 00:00:00  1.82558  0.42126  1.68367  0.33657  1.92460  0.12622  0.12799 -0.13459 -0.02820 -0.01496  0.22561  0.27225  0.36983 -0.75475
2  szse.000063  2020-01-02 00:00:00  0.26369 -0.06727  0.02359  0.12774  0.23008  0.49199  0.45774 -0.10291  0.91954 -1.04746 -0.45452 -0.07897  0.82966  0.14042
3  szse.000069  2020-01-02 00:00:00 -0.02552  0.27186 -0.12104 -0.24208 -0.11378 -0.24881 -0.63927 -0.33143 -0.29598 -0.38282  0.15367 -0.82864 -0.18707  0.52016
4  szse.000100  2020-01-02 00:00:00  0.77161  0.57301  0.94781 -0.71816  0.27502 -0.59353 -0.73266 -1.27401 -0.73786 -0.75526 -0.54192  0.59024 -0.71325  0.13904
Index(['code', 'date', 'macd

In [ ]:
# # 保存合并后的因子数据到CSV文件
# pd.DataFrame(merged_factors).to_csv('merged_factors.csv', index=False)

In [ ]:
import pandas as pd

# 合并因子数据和收益率数据
df_left = stock_long.copy()  # 左边数据集（收盘价和标签）
df_right = merged_factors.copy()  # 右边数据集（因子数据）

#  处理左边数据的时间：去掉时间部分，保留日期
df_left['date'] = pd.to_datetime(df_left['date']).dt.date

#  处理右边数据的时间（如果是字符串格式）
df_right['date'] = pd.to_datetime(df_right['date']).dt.date

#  排除2024-12-31的数据
df_left = df_left[df_left['date'] != '2024-12-31']
df_right = df_right[df_right['date'] != '2024-12-31']

#  按日期和股票代码合并
merged_df = pd.merge(
    df_left, 
    df_right,
    on=['date', 'code'],          # 合并键
    suffixes=('_left', '_right'), # 列名前缀
    how='inner'                   # 根据需求选连接方式
)



In [ ]:
# # 保存结果
# merged_df.to_csv('merged_data.csv', index=False)